# Backtest Analysis: Walk-Forward, Cross-Validation & Robustness

**Learning objectives:**
- Understand why simple backtest results are often misleading
- Run walk-forward analysis to test strategy robustness
- Perform parameter sensitivity analysis
- Analyze strategy performance across market regimes
- Test transaction cost sensitivity

This notebook demonstrates how to properly validate a trading strategy.

## 1. Setup

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path

project_root = Path('/Users/zelin/Desktop/PA Investment/Invest_strategy')
sys.path.insert(0, str(project_root))

import backtrader as bt
from backend.backtest_engine import BacktestEngine, IBKRDataFeed
from backtests.walkforward import WalkForwardAnalyzer, GridSearch, RegimeAnalyzer, CostSensitivityAnalyzer

print("Setup complete")

## 2. Load Data

In [ ]:
# Load equity data
ticker = 'SPY'
start_date = '2015-01-01'
end_date = '2024-12-31'

df = yf.download(ticker, start=start_date, end=end_date, progress=False)
df = df.droplevel('Ticker', axis=1)
df = df.reset_index()
df.columns = ['date', 'open', 'high', 'low', 'close', 'volume']

print(f"Loaded {len(df)} days of {ticker} data")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

## 3. Define Strategy Factory

In [ ]:
def create_momentum_strategy(params: dict = None):
    """Factory function to create momentum strategy."""
    params = params or {}
    period = params.get('period', 20)
    threshold = params.get('threshold', 0.0)
    
    class MomentumStrategy(bt.Strategy):
        params = (('period', period), ('threshold', threshold))
        
        def __init__(self):
            self.sma = bt.ind.SMA(self.data.close, period=self.params.period)
            self.order = None
        
        def next(self):
            if self.order:
                return
            sma_value = self.sma[-1]
            current_price = self.data.close[-1]
            
            if current_price > sma_value + self.params.threshold:
                if not self.position:
                    self.order = self.buy()
            elif current_price < sma_value - self.params.threshold:
                if self.position:
                    self.order = self.sell()
    
    MomentumStrategy.__name__ = 'MomentumStrategy'
    return MomentumStrategy

def create_mean_reversion_strategy(params: dict = None):
    """Factory function to create mean reversion strategy."""
    params = params or {}
    period = params.get('period', 20)
    std_dev = params.get('std_dev', 2.0)
    
    class MeanReversionStrategy(bt.Strategy):
        params = (('period', period), ('std_dev', std_dev))
        
        def __init__(self):
            self.sma = bt.ind.SMA(self.data.close, period=self.params.period)
            self.std = bt.ind.StandardDeviation(self.data.close, period=self.params.period)
            self.order = None
        
        def next(self):
            if self.order:
                return
            upper_band = self.sma[-1] + self.params.std_dev * self.std[-1]
            lower_band = self.sma[-1] - self.params.std_dev * self.std[-1]
            current_price = self.data.close[-1]
            
            if current_price > upper_band:
                if not self.position:
                    self.order = self.sell()
            elif current_price < lower_band:
                if not self.position:
                    self.order = self.buy()
            elif current_price > self.sma[-1] and self.position:
                self.order = self.sell()
    
    MeanReversionStrategy.__name__ = 'MeanReversionStrategy'
    return MeanReversionStrategy

print("Strategy factories defined")

## 4. Simple Backtest (The Problem)

In [ ]:
# First, let's run a simple backtest
engine = BacktestEngine(cash=100000, commission=0.001)
data_feed = IBKRDataFeed(dataname=df)
engine.add_data(data_feed, name='SPY')
engine.add_strategy(create_momentum_strategy({'period': 50}))

result = engine.run_backtest()

print("=== SIMPLE BACKTEST RESULTS ===")
print(f"Total Return: {result['total_return']*100:.2f}%")
print(f"Sharpe Ratio: {result['sharpe_ratio']:.2f}")
print(f"Max Drawdown: {result['max_drawdown']*100:.2f}%")
print(f"Total Trades: {result['total_trades']}")

print("
⚠️  WARNING: These results are likely OVERFITTED to this specific period!")
print("We need walk-forward analysis to validate robustness.")

## 5. Grid Search with Cross-Validation

In [ ]:
# Data loader function for walk-forward analyzer
def data_loader(start: str, end: str) -> pd.DataFrame:
    """Load data for given date range."""
    data = yf.download('SPY', start=start, end=end, progress=False)
    data = data.droplevel('Ticker', axis=1)
    return data

# Run grid search
print("Running grid search...")
gs = GridSearch(
    engine_class=BacktestEngine,
    strategy_factory=create_momentum_strategy,
    data=df,
    metric='sharpe_ratio'
)

param_grid = {
    'period': [10, 20, 30, 50, 100],
    'threshold': [0.0, 0.01, 0.02],
}

gs_result = gs.search(param_grid)

print("
=== GRID SEARCH RESULTS ===")
print(f"Best params: {gs_result.best_params}")
print(f"Best Sharpe: {gs_result.best_score:.2f}")

# Show top 5 results
results_df = gs_result.results_df
results_df = results_df.sort_values('score', ascending=False)
print("
Top 5 parameter combinations:")
print(results_df[['period', 'threshold', 'score', 'total_return', 'max_drawdown']].head())

## 6. Walk-Forward Analysis

In [ ]:
# Run walk-forward analysis
print("Running walk-forward analysis...")
print("(This may take a few minutes...)
")

wfa = WalkForwardAnalyzer(
    engine_class=BacktestEngine,
    data_loader=data_loader,
    strategy_factory=create_momentum_strategy,
    metric='sharpe_ratio',
    param_space={'period': [20, 30, 50], 'threshold': [0.0, 0.01]},
)

wf_summary = wfa.run(
    start_date='2015-01-01',
    end_date='2024-12-31',
    train_window=252 * 2,  # 2 years train
    test_window=63,        # 3 months test
    step=63,               # Roll forward every 3 months
    optimize=True,
)

print("
=== WALK-FORWARD RESULTS ===")
print(f"Number of iterations: {len(wf_summary.iterations)}")
print(f"Average Train Sharpe: {wf_summary.avg_train_sharpe:.2f}")
print(f"Average Test Sharpe: {wf_summary.avg_test_sharpe:.2f}")
print(f"Average Test Return: {wf_summary.avg_test_return*100:.2f}%")
print(f"Hit Rate (% of tests with + Sharpe): {wf_summary.hit_rate*100:.1f}%")
print(f"Sharpe Consistency (lower = more variable): {wf_summary.sharpe_consistency:.2f}")

## 7. Regime Analysis

In [ ]:
# Run simple backtest to get result
engine = BacktestEngine(cash=100000, commission=0.001)
engine.add_data(IBKRDataFeed(dataname=df), name='SPY')
engine.add_strategy(create_momentum_strategy({'period': 50}))
result = engine.run_backtest()

# Regime analysis
ra = RegimeAnalyzer(strategy_result=result, market_data=df)
regime_metrics = ra.analyze()

print("=== REGIME ANALYSIS ===
")
print("Performance by Trend Regime:")
print(f"  Bull Market:")
if 'trend_bull' in regime_metrics:
    m = regime_metrics['trend_bull']
    print(f"    Sharpe: {m.get('sharpe_ratio', 0):.2f}, Return: {m.get('total_return', 0)*100:.1f}%")
print(f"  Bear Market:")
if 'trend_bear' in regime_metrics:
    m = regime_metrics['trend_bear']
    print(f"    Sharpe: {m.get('sharpe_ratio', 0):.2f}, Return: {m.get('total_return', 0)*100:.1f}%")

print("
Performance by Volatility Regime:")
print(f"  High Vol:")
if 'vol_high_vol' in regime_metrics:
    m = regime_metrics['vol_high_vol']
    print(f"    Sharpe: {m.get('sharpe_ratio', 0):.2f}, Return: {m.get('total_return', 0)*100:.1f}%")
print(f"  Low Vol:")
if 'vol_low_vol' in regime_metrics:
    m = regime_metrics['vol_low_vol']
    print(f"    Sharpe: {m.get('sharpe_ratio', 0):.2f}, Return: {m.get('total_return', 0)*100:.1f}%")

## 8. Transaction Cost Sensitivity

In [ ]:
# Test how strategy performs under different cost levels
csa = CostSensitivityAnalyzer(
    engine_class=BacktestEngine,
    strategy_factory=create_momentum_strategy,
    data=df,
)

cost_results = csa.run()

print("=== TRANSACTION COST SENSITIVITY ===
")
print(cost_results.to_string(index=False))

# Find break-even cost
break_even = cost_results[cost_results['sharpe_ratio'] < 0].iloc[0]['commission_bps'] if len(cost_results[cost_results['sharpe_ratio'] < 0]) > 0 else None
if break_even:
    print(f"
⚠️  Break-even cost level: ~{break_even:.0f} bps")
else:
    print("
✓ Strategy remains profitable even at 100 bps")

## 9. Summary & Interpretation

In [ ]:
print("""
=== KEY TAKEAWAYS ===

1. **Simple Backtest is Overfitted**
   - The initial Sharpe is likely inflated due to overfitting
   - Never trust a backtest without out-of-sample validation

2. **Walk-Forward Analysis**
   - Tests strategy on rolling windows of unseen data
   - Hit rate > 60% indicates robustness
   - Low Sharpe consistency = unstable strategy

3. **Regime Analysis**
   - Strategies often work in one regime, fail in another
   - Know when your strategy will likely fail

4. **Cost Sensitivity**
   - High-turnover strategies are vulnerable to cost increases
   - Ensure margin of safety above break-even costs

5. **Next Steps**
   - Use Signal Framework to test multiple factors
   - Build Multi-Asset Portfolio to combine signals
""")